# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os
import numpy as np
import pandas as pd

# Work from the repo root whether this runs locally or fresh in Colab
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.system("git clone https://github.com/PrathamDudani/FlyRank_Assignment.git")
    os.chdir("FlyRank_Assignment")

pd.set_option("display.max_columns", 60)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)


(30000, 44)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The paper (`docs/flyrank-seo-research-march-2026.pdf`) already does a lot of this discipline itself —
it flags small samples and survivorship risk in its own text. In that same spirit, here are two
findings I'd want to ask the authors about, and why.

### Finding A — "The Freshness Multiplier" (growth-to-decline ratio by freshness window)

The paper reports a 31-90 day freshness window as the strongest stable growth signal (about 7.9
growing pages for every declining one), and separately flags that its oldest bucket (361+ days)
shows a much larger ratio built from a very small number of declining pages.

**My methodology question:** where does "growing" vs "declining" come from, and how many rows sit
behind each ratio? A ratio built on one declining page can swing wildly if that single row is an
outlier or a data artifact — the paper itself calls this bucket unstable, which is the right
instinct. The question I'd ask the authors: is there a minimum-row-count rule applied consistently
across *all* buckets (not just the one they happened to flag), so a reader can trust every ratio in
the table equally rather than needing the authors' running commentary to know which ones are solid?

### Finding B — ML Appendix, "What Predicts Growth?" (logistic regression, 71% holdout accuracy)

This model separates growing from declining pages using features like content age, days since
update, and days visible, evaluated on a holdout split.

**My methodology question:** is that holdout split a random row split, or is it grouped by brand
(the paper covers 57 brands)? If it's a random split, rows from the same brand could appear in both
the train and holdout sets, letting the model partly memorize brand-level quirks instead of
learning a pattern that generalizes to a brand it has never seen — exactly the failure mode this
assignment asks us to test for in our own model in Section 2. A second, related question: the
growth/decline label is built from a 30-day-vs-previous-30-day comparison — are any of the input
features (e.g. recent impressions) drawn from a window that overlaps that same comparison window?
If so, the model may be partly reading its own label rather than predicting it.

Both questions are asked in the constructive spirit the paper itself models: it already discloses
its evidence standard, keeps its ML pages labeled "exploratory," and flags its own small-sample
risk. These are the same two checks (group-aware split, window overlap) I run on my own model
below — turnabout is fair play.


In [3]:
# A small, concrete illustration of the "how many rows back this ratio" question,
# run on data I actually have access to (the starter CSV), not the paper's warehouse.
# This shows what the fragility the paper flags actually looks like in row counts.
bucket_counts = df["freshness_tier"].value_counts(dropna=False)
print("Row counts behind each freshness_tier bucket in MY dataset:")
print(bucket_counts)
print(
    "\nAny bucket with a small count here would produce a similarly shaky ratio -- "
    "this is the same check I'd want applied consistently in the paper's freshness table."
)


Row counts behind each freshness_tier bucket in MY dataset:
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64

Any bucket with a small count here would produce a similarly shaky ratio -- this is the same check I'd want applied consistently in the paper's freshness table.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score

valid = df[df["avg_position"] > 0].copy()
valid["is_declining_label"] = (valid["trend_direction"] == "down").astype(int)

features = ["ctr", "avg_position", "impressions_90d", "engagement_rate",
            "days_since_last_update", "search_volume"]


def fit_eval(train, test, label):
    X_train, y_train = train[features].fillna(0), train["is_declining_label"]
    X_test, y_test = test[features].fillna(0), test["is_declining_label"]
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        "split": label,
        "precision": round(precision_score(y_test, pred), 3),
        "recall": round(recall_score(y_test, pred), 3),
        "auc": round(roc_auc_score(y_test, proba), 3),
        "base_rate": round(y_test.mean(), 3),
        "n_test": len(test),
    }, model


# BEFORE -- naive random row split, no grouping
train_naive, test_naive = train_test_split(
    valid, test_size=0.2, random_state=42, stratify=valid["is_declining_label"]
)
before, _ = fit_eval(train_naive, test_naive, "naive random split (BEFORE)")

# AFTER -- honest split, grouped by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(valid, groups=valid["client_id"]))
train_grouped, test_grouped = valid.iloc[train_idx], valid.iloc[test_idx]
after, model_after = fit_eval(train_grouped, test_grouped, "client-grouped split (AFTER)")

train_clients = set(train_naive["client_id"])
test_clients = set(test_naive["client_id"])
overlap = len(train_clients & test_clients)

results = pd.DataFrame([before, after])
print(results.to_string(index=False))
print(f"\nClients present in BOTH naive train and test: {overlap} of {valid['client_id'].nunique()}")


                       split  precision  recall   auc  base_rate  n_test
 naive random split (BEFORE)      0.602   0.652 0.566      0.565    5759
client-grouped split (AFTER)      0.576   0.667 0.542      0.540    5821

Clients present in BOTH naive train and test: 30 of 31


**Observed, not dramatic — and that's itself the finding.** Moving from the naive random
split to the client-grouped split drops AUC and precision only slightly (both stayed close to the
base rate either way, on this six-feature model). What the naive split does clearly expose is the
mechanism: nearly every client in the filtered dataset showed up in both its train and test sets
(see the exact count printed above), so a random split was never actually testing "does this
generalize to a client the model hasn't seen" — the honest question this task asks about. The grouped split is the correct default here regardless of how big the
before/after gap happens to be, because it's the split that matches how the model would actually be
used (scoring a client's content, including new clients).

For contrast: the reference pipeline in this repo (`scripts/03_train_model.py` →
`outputs/model_report.md`) reports a materially higher AUC (0.70-0.75) using a richer feature set
(log-transformed volume features, `days_with_impressions`, categorical fields) and the same
client-aware split logic. That gap is a feature-set difference, not a split-honesty difference —
worth keeping straight when comparing numbers across notebooks in this repo (see Section 4).


In [5]:
# Real failure examples from the HONEST split (client-grouped), not the naive one.
# A big metric table hides what the model actually gets wrong -- look at the rows.
pred_after = model_after.predict(test_grouped[features].fillna(0))
proba_after = model_after.predict_proba(test_grouped[features].fillna(0))[:, 1]

errors = test_grouped.copy()
errors["pred"] = pred_after
errors["proba"] = proba_after
errors["true"] = errors["is_declining_label"]

false_positives = errors[(errors["pred"] == 1) & (errors["true"] == 0)]
false_negatives = errors[(errors["pred"] == 0) & (errors["true"] == 1)]

print(f"False positives (flagged as declining, actually not): {len(false_positives)} of {len(test_grouped)} test rows")
print(f"False negatives (missed an actual decline):           {len(false_negatives)} of {len(test_grouped)} test rows")

cols_to_show = ["client_id"] + features + ["proba", "true", "pred"]
print("\nFive example false positives (model confident, model wrong):")
print(false_positives.sort_values("proba", ascending=False)[cols_to_show].head(5).to_string(index=False))

print("\nFive example false negatives (model confident it was fine, actually declining):")
print(false_negatives.sort_values("proba")[cols_to_show].head(5).to_string(index=False))


False positives (flagged as declining, actually not): 1546 of 5821 test rows
False negatives (missed an actual decline):           1046 of 5821 test rows

Five example false positives (model confident, model wrong):
        client_id  ctr  avg_position  impressions_90d  engagement_rate  days_since_last_update  search_volume    proba  true  pred
client_4ec9599fc2  0.0           5.0               10             0.00                     334            NaN 0.768237     0     1
client_4ec9599fc2  0.0           8.9              103             0.00                     304            NaN 0.740224     0     1
client_4ec9599fc2  0.0          35.0                1             0.00                     372            0.0 0.716370     0     1
client_4ec9599fc2  0.0          20.4               64             0.00                     301            NaN 0.707895     0     1
client_8722616204  0.0           4.5                2             6.25                     231          720.0 0.695519     0     

**What the failures actually look like** (read from the printed rows above, not guessed in
advance): the most-confident false positives are stale, near-zero-signal pages — `ctr` of 0.0,
very few `impressions_90d` (as low as 1-100), and `days_since_last_update` in the 300+ day range.
The model reads "stale and quiet" as "declining," but plenty of quiet pages are just small and
stable, not falling. The most-confident false negatives are the opposite profile: pages with huge
`impressions_90d` (300K-500K), good `avg_position` (under 5), and recently updated
(`days_since_last_update` of 20-104 days) — a page that looks strong on every visible number can
still be the one that's actually declining, and the model's `class_weight="balanced"` setting
isn't enough to catch that pattern with only six features and no trend-shape input. Practical read:
this model is currently better at spotting "quiet and untouched" than at catching decline in
otherwise-strong, actively-updated pages — exactly the pages a content team would most want a
warning about. A reviewer should not treat a low probability score as reassurance on a
high-traffic page.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# 1) Where does the label actually come from? Confirm the formula mechanically
#    rather than trusting the data dictionary's description.
check = df[["impressions_last_30d", "impressions_prev_30d", "trend_pct"]].copy()
check["recomputed_pct"] = np.where(
    check["impressions_prev_30d"] > 0,
    (check["impressions_last_30d"] - check["impressions_prev_30d"])
    / check["impressions_prev_30d"] * 100,
    np.nan,
)
formula_corr = check[["trend_pct", "recomputed_pct"]].corr().iloc[0, 1]
print("Correlation between published trend_pct and the recomputed 30d-vs-prev-30d change:",
      round(formula_corr, 4))
print("--> trend_direction (and therefore is_declining_label) is a direct bucketing of this "
      "value. impressions_last_30d, impressions_prev_30d, and trend_pct are label-derived and "
      "must never be features -- confirmed, not assumed.")


Correlation between published trend_pct and the recomputed 30d-vs-prev-30d change: 1.0
--> trend_direction (and therefore is_declining_label) is a direct bucketing of this value. impressions_last_30d, impressions_prev_30d, and trend_pct are label-derived and must never be features -- confirmed, not assumed.


In [7]:
# 2) The "add a leaky feature and watch the score jump" test -- proves the test
#    harness itself can detect leakage before we trust any "no leakage found" result.
leaky_features = features + ["impressions_last_30d", "impressions_prev_30d", "trend_pct"]

X_train_leak = train_grouped[leaky_features].fillna(0)
X_test_leak = test_grouped[leaky_features].fillna(0)
model_leak = LogisticRegression(max_iter=1000, class_weight="balanced")
model_leak.fit(X_train_leak, train_grouped["is_declining_label"])
proba_leak = model_leak.predict_proba(X_test_leak)[:, 1]
auc_leak = round(roc_auc_score(test_grouped["is_declining_label"], proba_leak), 3)

print(f"Honest AUC, clean features (same split as Section 2 'AFTER'): {after['auc']}")
print(f"AUC with the label-derived columns deliberately added back in:  {auc_leak}")
print("--> the jump confirms the harness would catch this leak. The clean features (Section 2) "
      "are the number to trust; the leaky run above is discarded, not reported as a result.")


Honest AUC, clean features (same split as Section 2 'AFTER'): 0.542
AUC with the label-derived columns deliberately added back in:  1.0
--> the jump confirms the harness would catch this leak. The clean features (Section 2) are the number to trust; the leaky run above is discarded, not reported as a result.


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
# 3) Window-overlap check: impressions_90d is a 90-day sum that CONTAINS the two
#    30-day windows the label is built from (last_30d vs prev_30d). A stricter feature
#    would only use the days BEFORE that comparison window.
valid["impressions_earlier_period"] = (
    valid["impressions_90d"] - valid["impressions_last_30d"] - valid["impressions_prev_30d"]
).clip(lower=0)

features_clean_window = [f if f != "impressions_90d" else "impressions_earlier_period"
                          for f in features]

train_c, test_c = valid.iloc[train_idx], valid.iloc[test_idx]  # identical grouped split
model_clean = LogisticRegression(max_iter=1000, class_weight="balanced")
model_clean.fit(train_c[features_clean_window].fillna(0), train_c["is_declining_label"])
proba_clean = model_clean.predict_proba(test_c[features_clean_window].fillna(0))[:, 1]
auc_window_clean = round(roc_auc_score(test_c["is_declining_label"], proba_clean), 3)

print(f"AUC using impressions_90d (overlaps the label's comparison window):        {after['auc']}")
print(f"AUC using only impressions from BEFORE that comparison window:             {auc_window_clean}")
print("--> the two numbers land close together, so this particular overlap isn't quietly "
      "inflating the result on this feature set. Worth re-checking any time a new 90-day-style "
      "aggregate is added, since the window still technically overlaps the label by construction.")


AUC using impressions_90d (overlaps the label's comparison window):        0.542
AUC using only impressions from BEFORE that comparison window:             0.523
--> the two numbers land close together, so this particular overlap isn't quietly inflating the result on this feature set. Worth re-checking any time a new 90-day-style aggregate is added, since the window still technically overlaps the label by construction.


In [9]:
# 4) Decision-derived features / product flags: confirm none are in the feature list.
#    (This starter export doesn't ship a FlyRank "health score" or optimization-flag
#    column at all -- that construct lives only in the paper's separate warehouse data.)
suspect_columns = [c for c in df.columns if "flag" in c.lower() or "score" in c.lower()]
print("Columns in this dataset that look like an existing system's decision output:",
      suspect_columns or "none found")


Columns in this dataset that look like an existing system's decision output: none found


**Attack checklist, filled in:**

- [x] Timeline drawn: features are 90-day aggregates or static fields; the label is built from the
  final two 30-day slices of that same window — flagged and tested in cell 3 above (no dramatic
  collapse found on this feature set, but the overlap is structural and worth re-checking).
- [x] No label-derived or sibling columns in the features: `trend_direction`, `trend_pct`,
  `impressions_last_30d`, `impressions_prev_30d` all excluded. Confirmed with the deliberate-leak
  test in cell 2 (AUC jumps once they're added back).
- [x] No product flags / existing-system scores as features: none exist in this export (cell 4).
- [x] Split grouped by the repeating entity (`client_id`) — Section 2.
- [x] Base rate printed next to every metric — Section 2 table (`base_rate` column).
- [x] Top feature behavior sanity-checked: none of the six features shows the "one feature towers
  over the rest" symptom that usually signals a hidden leak (see Week-5 notebook's permutation
  importances — the top feature explains well under half of total importance).
- [x] Metrics recomputed out-of-fold: all numbers above are held-out test performance, never
  in-sample.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (from this repo's `README.md`, describing the reference pipeline):**

> the learned model clearly beats the hand-written rule at picking the right pages to review first
> (Precision@50 ≈ 0.24 → 0.74 ... the ~3x lift is the point)

This reads as a settled, general fact about "the model." Sections 2 and 3 above give a more honest
basis for it, so here is the rewrite:

**Rewritten:** In one client-grouped holdout run on the bundled sample, the reference model's
precision among its top-50 ranked items was measured at roughly 0.74, versus roughly 0.24 for the
rule-based baseline on the same split — an observed, directional lift on this metric and this slice
of the ranking. It is decision-support evidence for prioritizing the top of the queue, not a
guarantee: the same repo's simpler six-feature model (Section 2 above) scores much closer to the
base rate on overall AUC, precision@50 is sensitive to which 50 rows land on top in a given run, and
neither number has been shown to hold on a brand the model has never scored before.

---

**A second claim**, from `outputs/model_report.md`'s top-feature list (`days_with_impressions:
0.1578` listed as the top feature), is easy to over-read as "the reason a page declines."

**Rewritten:** `days_with_impressions` was observed to carry the largest feature-importance weight
in one random-forest run on this sample — a descriptive statement about what that particular model
leaned on, not a causal claim about what drives decline, and not yet checked against a
grouped-split permutation-importance run of its own.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.